# Diagnostic Tests I — Heteroskedasticity & Autocorrelation — Lecture Notebook
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW
**Based on:** Brooks, C. — *Introductory Econometrics for Finance*, Cambridge University Press, Ch. 5

---
**Learning Objectives:**
- **Spot** heteroskedasticity (fan, volatility clusters) and autocorrelation (residual waves) in residual plots
- **Test** formally: **Goldfeld-Quandt**, **Breusch-Pagan**, **White** (variance) and **Durbin-Watson**, **Breusch-Godfrey** (correlation)
- **Fix** the inference: robust **HC** and **Newey-West (HAC)** standard errors
- **Fix** the model: add lag dynamics, choose the lag length with **information criteria (AIC / BIC)**

> Run each cell with **Shift+Enter**. This notebook accompanies the V8 lecture slides.
> The slide numbers use illustrative values; the code below computes the *live* numbers from current data — they will be close, but not identical.

## Step 0 — Install & Import Libraries

In [ ]:
!pip install yfinance pandas-datareader statsmodels --quiet

import yfinance as yf
import pandas_datareader.data as web
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, het_white, het_goldfeldquandt, acorr_breusch_godfrey
from statsmodels.stats.stattools import durbin_watson
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.3, 'font.size':11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'; GREEN = '#0B7A3C'
print('✓ Libraries loaded.')

---
# Part 1 — The Two Running Examples

Diagnostics only make sense on a fitted model, so we set up both regressions from the earlier chapters:

1. **Apple CAPM** (daily *returns*) — our heteroskedasticity patient: $r_{AAPL,t} = \beta_0 + \beta_1 r_{Mkt,t} + u_t$
2. **Mortgage pass-through** (weekly *levels*) — our autocorrelation patient: $mort_t = \beta_0 + \beta_1 y10_t + u_t$

### 1.1 Apple CAPM — daily returns

In [ ]:
START, END = '2020-01-01', '2024-12-31'

px  = yf.download(['AAPL', '^GSPC'], start=START, end=END,
                  auto_adjust=True, progress=False)['Close']
ret = px.pct_change().dropna()
# rename by label, never by position: yfinance orders the Close columns alphabetically
ret = ret.rename(columns={'^GSPC': 'SP500'})[['AAPL', 'SP500']]

X_capm = sm.add_constant(ret['SP500'])
capm   = sm.OLS(ret['AAPL'], X_capm).fit()

print(f'Apple CAPM: n = {len(ret)} trading days')
print(f'beta_hat = {capm.params["SP500"]:.4f}   SE = {capm.bse["SP500"]:.4f}   R² = {capm.rsquared:.3f}')

### 1.2 Mortgage pass-through — weekly levels

In [ ]:
m30 = web.DataReader('MORTGAGE30US', 'fred', '2010-01-01', '2024-12-31')  # weekly
y10 = web.DataReader('DGS10',        'fred', '2010-01-01', '2024-12-31')  # daily

lvl = m30.join(y10.resample('W-THU').mean(), how='inner').dropna()
# rename by label, never by position
lvl = lvl.rename(columns={'MORTGAGE30US': 'mort', 'DGS10': 'y10'})[['mort', 'y10']]

X_mort = sm.add_constant(lvl['y10'])
mort   = sm.OLS(lvl['mort'], X_mort).fit()

print(f'Mortgage pass-through: n = {len(lvl)} weeks')
print(f'beta_hat (pass-through) = {mort.params["y10"]:.4f}   SE = {mort.bse["y10"]:.4f}   R² = {mort.rsquared:.3f}')

**Remember what is at stake:** if the errors violate A2 (constant variance) or A3 (no autocorrelation), these betas stay **unbiased** — but the printed standard errors, t-statistics and p-values are **wrong**, typically too optimistic.

---
# Part 2 — Eyes First: Residual Plots

Always plot the residuals before any formal test. We are looking for:
- the **fan** (spread grows with the fitted value) → heteroskedasticity
- **volatility clusters** (calm vs. turbulent episodes) → heteroskedasticity in time
- smooth **waves** (residual sticks to its own past) → autocorrelation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

# Apple: residual vs fitted — look for the fan
axes[0,0].scatter(capm.fittedvalues*100, capm.resid*100, s=6, color=RED, alpha=0.4)
axes[0,0].axhline(0, color='black', lw=1)
axes[0,0].set_title('Apple CAPM: residual vs. fitted — fan?', fontweight='bold', loc='left')
axes[0,0].set_xlabel('fitted (%)'); axes[0,0].set_ylabel('residual (%)')

# Apple: residual over time — volatility clusters
axes[0,1].plot(ret.index, capm.resid*100, color=GREY, lw=0.6)
axes[0,1].axhline(0, color=RED, lw=1)
axes[0,1].set_title('Apple CAPM: residual over time — clusters?', fontweight='bold', loc='left')

# Mortgage: residual over time — waves
axes[1,0].plot(lvl.index, mort.resid, color=GREY, lw=1.2)
axes[1,0].axhline(0, color=RED, lw=1)
axes[1,0].set_title('Mortgage levels: residual over time — WAVES', fontweight='bold', loc='left')

# Mortgage: residual vs its own lag — the autocorrelation scatter
axes[1,1].scatter(mort.resid.shift(1), mort.resid, s=8, color=BLUE, alpha=0.5)
axes[1,1].set_title('Mortgage: residual vs. lagged residual', fontweight='bold', loc='left')
axes[1,1].set_xlabel('û(t−1)'); axes[1,1].set_ylabel('û(t)')

plt.tight_layout(); plt.show()
rho1 = mort.resid.autocorr(1)
print(f'First-order residual autocorrelation (mortgage): rho_hat = {rho1:.3f}')

---
# Part 3 — Goldfeld-Quandt: Split and Compare

**Recipe:** order the observations by the suspected variance driver (here: $|r_{Mkt}|$), drop the middle band (~17%) to sharpen the contrast, run OLS on each half, and compare residual variances:

$$GQ = \frac{RSS_2/df_2}{RSS_1/df_1} \sim F_{df_2,\,df_1} \text{ under } H_0$$

### 3.1 By hand

In [ ]:
order = ret['SP500'].abs().sort_values().index   # low |x| → high |x|
n     = len(order)
drop  = round(0.17 * n)                          # middle band, ~17% of the sample
half  = (n - drop) // 2

low_idx, high_idx = order[:half], order[-half:]

m_low  = sm.OLS(ret.loc[low_idx,  'AAPL'], sm.add_constant(ret.loc[low_idx,  'SP500'])).fit()
m_high = sm.OLS(ret.loc[high_idx, 'AAPL'], sm.add_constant(ret.loc[high_idx, 'SP500'])).fit()

s2_low, s2_high = m_low.ssr/m_low.df_resid, m_high.ssr/m_high.df_resid
GQ = s2_high / s2_low
F_crit = stats.f.ppf(0.95, int(m_high.df_resid), int(m_low.df_resid))

print(f'n = {n}, dropped middle = {drop}, days per half = {half}, df per half = {int(m_low.df_resid)}')
print(f's² (low-|x| half)  = {s2_low:.3e}')
print(f's² (high-|x| half) = {s2_high:.3e}')
print(f'GQ = {GQ:.2f}   vs   F_crit(5%) = {F_crit:.2f}')
print('→ REJECT homoskedasticity' if GQ > F_crit else '→ do not reject')

### 3.2 Verify with statsmodels

In [ ]:
# het_goldfeldquandt needs the data ORDERED by the variance driver
ord_idx = ret['SP500'].abs().sort_values().index
# statsmodels is changing what these tests return: from release 0.16 on,
# acorr_breusch_godfrey and het_goldfeldquandt hand back a named result object
# instead of a plain tuple. Positional INDEXING works in both worlds, fixed-arity
# unpacking does not, so read element 0 (statistic) and element 1 (p-value).
gq = het_goldfeldquandt(ret.loc[ord_idx, 'AAPL'],
                        sm.add_constant(ret.loc[ord_idx, 'SP500']),
                        split=0.415, drop=0.17)   # 41.5% + 17% + 41.5%
GQ_sm, p_sm = gq[0], gq[1]
print(f'statsmodels: GQ = {GQ_sm:.2f},  p = {p_sm:.2e}')
print('\nCaveat: YOU chose the ordering variable and the split — both somewhat arbitrary.')
print('That is why GQ is only the opener; the regression-based tests need no such choices.')

---
# Part 4 — Breusch-Pagan & White: Auxiliary Regressions

**The idea:** if the variance is constant, NOTHING should explain $\hat{u}^2$. Regress the squared residuals on the regressors and look at $R^2_{aux}$:

$$LM = n \cdot R^2_{aux} \sim \chi^2_{(\text{# variance drivers})}$$

*LM stands for Lagrange Multiplier — the test principle that only needs the model estimated under $H_0$.*

### 4.1 Breusch-Pagan by hand

In [ ]:
aux = sm.OLS(capm.resid**2, X_capm).fit()      # û² on the regressors
LM  = len(ret) * aux.rsquared
chi2_crit = stats.chi2.ppf(0.95, 1)

print(f'R²_aux = {aux.rsquared:.4f}  →  the market return explains '
      f'{aux.rsquared*100:.1f}% of the squared residuals')
print(f'LM = n · R²_aux = {len(ret)} × {aux.rsquared:.4f} = {LM:.1f}')
print(f'χ²_crit(1, 5%) = {chi2_crit:.2f}   →   ' + ('REJECT: heteroskedastic' if LM > chi2_crit else 'do not reject'))

### 4.2 One-line versions, plus White

In [ ]:
lm_bp, p_bp, _, _ = het_breuschpagan(capm.resid, capm.model.exog)
lm_w,  p_w,  _, _ = het_white(capm.resid, capm.model.exog)

print(f'Breusch-Pagan:  LM = {lm_bp:.1f},  p = {p_bp:.2e}')
print(f'White:          LM = {lm_w:.1f},  p = {p_w:.2e}   (adds squares → χ² with 2 df, crit 5.99)')
print('\nSame trick as the VIF: an auxiliary regression whose R² should be ≈ 0 under H0.')

---
# Part 5 — Fix 1 for Heteroskedasticity: Robust (HC) Standard Errors

Robust standard errors re-weight each observation's contribution by its own squared residual. **Point estimates unchanged — only the SEs are corrected.** One argument in Python.

In [ ]:
capm_hc = sm.OLS(ret['AAPL'], X_capm).fit(cov_type='HC1')   # or 'HC3' for small samples

cmp = pd.DataFrame({
    'beta_hat': [capm.params['SP500'],  capm_hc.params['SP500']],
    'SE':       [capm.bse['SP500'],     capm_hc.bse['SP500']],
    't':        [capm.tvalues['SP500'], capm_hc.tvalues['SP500']],
}, index=['OLS', 'HC1 robust']).round(4)
print(cmp)
print(f'\nSE inflation: ×{capm_hc.bse["SP500"]/capm.bse["SP500"]:.3f}')
print('Practice standard: in empirical finance, robust SEs are the DEFAULT, not the exception.')

---
# Part 6 — Autocorrelation: Durbin-Watson

$$DW \approx 2(1 - \hat{\rho})$$

DW ≈ 2 → no autocorrelation • DW → 0 → strong positive • DW → 4 → negative.

**Two caveats:** DW only sees lag-1 correlation, and it is **invalid with a lagged dependent variable** among the regressors — both solved by Breusch-Godfrey.

In [ ]:
dw_mort = durbin_watson(mort.resid)
dw_capm = durbin_watson(capm.resid)

print(f'Mortgage LEVELS regression:  DW = {dw_mort:.2f}  →  rho_hat ≈ {1 - dw_mort/2:.2f}')
print(f'Apple RETURN regression:     DW = {dw_capm:.2f}  →  rho_hat ≈ {1 - dw_capm/2:.2f}')
for name, dw in (('levels', dw_mort), ('returns', dw_capm)):
    rho = 1 - dw/2
    verdict = ('massive positive autocorrelation' if rho > 0.5 else
               'clear positive autocorrelation'   if rho > 0.2 else
               'almost none'                      if abs(rho) <= 0.1 else
               'negative autocorrelation')
    print(f'  {name:<8} rho_hat = {rho:+.2f}  ->  {verdict}')
print('Know your data type: levels of a rate carry the memory, returns mostly do not.')

---
# Part 7 — Breusch-Godfrey: The General Test

Auxiliary regression #3: regress $\hat{u}_t$ on the **original regressors AND its own $p$ lags**:

$$LM = (n-p) \cdot R^2_{aux} \sim \chi^2_p$$

Two versions of this statistic are in circulation: $(n-p)R^2_{aux}$, used below by hand and in Brooks, and $n\,R^2_{aux}$, which is what `statsmodels` returns. They differ by $p\,R^2_{aux}$ — a few units when $n$ is in the hundreds, never enough to change the decision; do not be surprised when the two printed values differ slightly.

The lag length $p$ is **your** choice (a convention, not a formula): with weekly data, $p=4$ covers roughly a month of error memory. The $\chi^2$ degrees of freedom equal $p$ — the number of tested restrictions $\rho_1 = \dots = \rho_p = 0$.

### 7.1 By hand

In [ ]:
p = 4
u = mort.resid
aux_df = pd.DataFrame({'y10': lvl['y10']})
for j in range(1, p+1):
    aux_df[f'u_lag{j}'] = u.shift(j)
aux_df['u'] = u
aux_df = aux_df.dropna()

aux_bg = sm.OLS(aux_df['u'], sm.add_constant(aux_df.drop(columns='u'))).fit()
LM_bg  = len(aux_df) * aux_bg.rsquared          # n − p usable observations
chi2_c = stats.chi2.ppf(0.95, p)

print(f'R²_aux = {aux_bg.rsquared:.3f}  →  the lags explain {aux_bg.rsquared*100:.0f}% of today’s residual')
print(f'LM = (n−p) · R²_aux = {len(aux_df)} × {aux_bg.rsquared:.3f} = {LM_bg:.1f}')
print(f'χ²_crit({p}, 5%) = {chi2_c:.2f}   →   ' + ('massive REJECTION: strong error memory' if LM_bg > chi2_c else 'do not reject'))

### 7.2 One line

In [ ]:
# statsmodels is changing what these tests return: from release 0.16 on,
# acorr_breusch_godfrey and het_goldfeldquandt hand back a named result object
# instead of a plain tuple. Positional INDEXING works in both worlds, fixed-arity
# unpacking does not, so read element 0 (statistic) and element 1 (p-value).
bg = acorr_breusch_godfrey(mort, nlags=4)
lm_bg, p_bg = bg[0], bg[1]
print(f'Breusch-Godfrey (4 lags): LM = {lm_bg:.1f},  p = {p_bg:.2e}')

---
# Part 8 — Fix 1 for Autocorrelation: Newey-West (HAC) Standard Errors

HAC = heteroskedasticity- **and** autocorrelation-consistent: the variance formula adds the first $L$ autocovariances of the residuals, with declining **Bartlett weights** $w_j = 1 - \frac{j}{L+1}$.

**Choosing the bandwidth $L$** — rule of thumb: $L \approx 0.75\,n^{1/3}$ (the allowance grows only with the *cube root* of the sample — doubling the data adds ~26% more lags). $L$ is a bandwidth of the *estimator*, not a model choice.

In [ ]:
n_m = len(lvl)
L   = int(np.ceil(0.75 * n_m**(1/3)))
print(f'Rule of thumb: L ≈ 0.75 × {n_m}^(1/3) = {0.75 * n_m**(1/3):.1f}  →  L = {L}')

mort_nw = sm.OLS(lvl['mort'], X_mort).fit(cov_type='HAC', cov_kwds={'maxlags': L})

cmp = pd.DataFrame({
    'beta_hat': [mort.params['y10'],  mort_nw.params['y10']],
    'SE':       [mort.bse['y10'],     mort_nw.bse['y10']],
    't':        [mort.tvalues['y10'], mort_nw.tvalues['y10']],
}, index=['OLS', f'Newey-West (L={L})']).round(4)
print(cmp)
flips = (abs(mort.tvalues['y10']) > 1.96) != (abs(mort_nw.tvalues['y10']) > 1.96)
print(f'\nSE inflation: ×{mort_nw.bse["y10"]/mort.bse["y10"]:.1f} — '
      + ('the conclusion FLIPS: what looked significant under OLS no longer is. '
         'That is exactly why the correction matters.' if flips else
         'same conclusion, honest uncertainty.'))

### 8.1 The Bartlett weights, visualised

In [ ]:
j = np.arange(1, L+1)
w = 1 - j/(L+1)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(j, w, color=YELLOW, edgecolor=GREY)
for ji, wi in zip(j, w):
    ax.text(ji, wi + 0.02, f'{wi:.3f}', ha='center', fontsize=9)
ax.set_xlabel('autocovariance lag j (weeks)'); ax.set_ylabel('weight w_j')
ax.set_title(f'Newey-West with L = {L}: recent error memory counts most', fontweight='bold', loc='left')
plt.tight_layout(); plt.show()

---
# Part 9 — Fix 2: Model the Dynamics (the Better Fix)

Autocorrelated residuals often mean the model **misses dynamics** — mortgage rates adjust over months, not within a week. Put the adjustment INTO the model:

$$mort_t = \alpha + \varphi_1\, mort_{t-1} + \dots + \varphi_p\, mort_{t-p} + \theta\, y10_t + u_t$$

**How many lags $p$? Let the information criteria decide** (Brooks' formulas, with $\hat{\sigma}^2 = RSS/T$):

$$AIC = \ln(\hat{\sigma}^2) + \frac{2k}{T} \qquad BIC = \ln(\hat{\sigma}^2) + \frac{k}{T}\ln T$$

*(Brooks writes SBIC; software output says BIC — same formula. A third criterion, Hannan-Quinn, sits between the two.)* Both reward fit and punish parameters; BIC punishes hardest since $\ln T > 2$. **Recipe: estimate $p = 0, 1, \dots, p_{max}$ and pick the minimum.**

### 9.1 The information-criteria table

In [ ]:
P_MAX = 4
# AIC and BIC are only comparable across models estimated on the SAME sample. Every
# extra lag would otherwise cost one observation at the start of the sample, which
# lowers RSS for mechanical reasons. Fix the estimation window at the first date the
# largest model (p = P_MAX) can use.
FIRST = lvl.index[P_MAX]

def dynamic_model(p):
    df = pd.DataFrame({'y10': lvl['y10']})
    for j in range(1, p+1):
        df[f'mort_lag{j}'] = lvl['mort'].shift(j)
    df['mort'] = lvl['mort']
    df = df.dropna().loc[FIRST:]          # identical sample for every p
    X = sm.add_constant(df.drop(columns='mort'))
    return sm.OLS(df['mort'], X).fit(), df

rows = []
for p in range(P_MAX + 1):
    m, df_p = dynamic_model(p)
    T, k = len(df_p), int(m.df_model) + 1            # k = number of estimated parameters
    sig2 = m.ssr / T
    rows.append({'lags p': p,
                 'AIC': np.log(sig2) + 2*k/T,
                 'BIC': np.log(sig2) + k/T*np.log(T)})
ic = pd.DataFrame(rows).set_index('lags p').round(4)
print(ic)
print(f'\nAIC minimum at p = {ic["AIC"].idxmin()},  BIC minimum at p = {ic["BIC"].idxmin()}')
print('The criteria MAY disagree — BIC is the more parsimonious (and consistent) choice.')

### 9.2 Estimate the chosen model and re-run the diagnostics

In [ ]:
p_star = int(ic['BIC'].idxmin())
m_dyn, df_dyn = dynamic_model(p_star)

phi   = m_dyn.params.filter(like='mort_lag').sum()   # total adjustment coefficient
theta = m_dyn.params['y10']

print(m_dyn.summary())
print(f'\nAdjustment (sum of lag coefficients): {phi:.3f}')
print(f'Short-run effect of the Treasury yield: {theta:.3f}')
print(f'Long-run pass-through: theta / (1 − phi) = {theta/(1-phi):.3f}')

In [ ]:
# Autocorrelation check on the DYNAMIC model.
# Careful: with a lagged y among the regressors DW is NO LONGER VALID (biased towards 2).
# Breusch-Godfrey remains valid — so BG is the test to quote here.
bg_dyn = acorr_breusch_godfrey(m_dyn, nlags=4)   # indexed, not unpacked
lm_dyn, p_dyn = bg_dyn[0], bg_dyn[1]
print(f'Breusch-Godfrey on the dynamic model: LM = {lm_dyn:.1f},  p = {p_dyn:.3f}')
print('→ clean' if p_dyn > 0.05 else '→ still autocorrelated — consider more lags')
print(f'(DW = {durbin_watson(m_dyn.resid):.2f} — reported only for illustration; do not test with it here.)')
print('\nSymptom vs. cause: Newey-West repairs the INFERENCE; the lags repair the MODEL.')

---
## Summary Table

| Problem | Spot | Test | Fix |
|---------|------|------|-----|
| Heteroskedasticity | fan, volatility clusters | GQ (opener), **BP / White**: $LM = n R^2_{aux} \sim \chi^2$ | `fit(cov_type='HC1')` |
| Autocorrelation | residual waves | **DW** ≈ 2(1−$\hat{\rho}$) (lag-1 only), **BG**: $(n{-}p) R^2_{aux} \sim \chi^2_p$ | `fit(cov_type='HAC', cov_kwds={'maxlags': L})` — or add lags |
| Lag length (model) | — | **AIC / BIC** minimum over $p$ | loop over `p`, pick min BIC |
| NW bandwidth | — | rule of thumb $L \approx 0.75\,n^{1/3}$ | `maxlags=L` |

**One idea, many diagnostics:** BP, White, BG (and the VIF) all run *auxiliary regressions* whose $R^2$ should be ≈ 0 under $H_0$.

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*

*Next: Diagnostic Tests II — Functional Form, Normality & Outliers.*